# Experimento 4: Diferencias temáticas por bloque político

**Pregunta:** ¿Se pueden identificar diferencias temáticas entre bloques o partidos políticos?

**Input:** `data/intervenciones_con_topico.parquet` + `data/diputados_historial_limpio.csv` + `data/bloques_familia_politica.csv`  
**Alcance temporal:** 2007–2026 (el historial de diputados comienza el 10/12/2007)  
**Output:** Heatmap de tópicos × familia política, test chi-cuadrado, TF-IDF por familia

## 1. Imports

In [35]:
import pandas as pd
import numpy as np
import unicodedata
import re
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from scipy.stats import chi2_contingency
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Cargar datos

In [36]:
# ── Intervenciones con tópico ─────────────────────────────────────────────────
df = pd.read_parquet("../data/intervenciones_limpias.parquet")
df["fecha"] = pd.to_datetime(df["fecha"])
df["anio"]  = df["fecha"].dt.year

# Asignar tópicos desde caché
df_top = pd.read_parquet(
    "../data/intervenciones_con_topico.parquet",
    columns=["id_periodo","id_reunion","n_intervencion","topic"]
).drop_duplicates(subset=["id_periodo","id_reunion","n_intervencion"])

topico_map = df_top.set_index(["id_periodo","id_reunion","n_intervencion"])["topic"]
df["topic"] = df.set_index(["id_periodo","id_reunion","n_intervencion"]).index.map(topico_map).values
del df_top, topico_map

# Filtrar al período con datos de bloques
df = df[df["anio"] >= 2007].copy()

# ── Historial de diputados y bloques ─────────────────────────────────────────
dh = pd.read_csv("../data/diputados_historial_limpio.csv")
dh["bloque_inicio"] = pd.to_datetime(dh["bloque_inicio"])
dh["bloque_fin"]    = pd.to_datetime(dh["bloque_fin"])

bf = pd.read_csv("../data/bloques_familia_politica.csv")
dh = dh.merge(bf, on="bloque", how="left")

print(f"Intervenciones post-2007: {len(df):,}")
print(f"Diputados en historial:   {dh['id'].nunique():,}")
print(f"Familias políticas:       {dh['familia_politica'].nunique()}")

Intervenciones post-2007: 87,339
Diputados en historial:   1,133
Familias políticas:       10


## 3. Normalización y JOIN orador → diputado

Estrategia:
1. Filtrar intervenciones procedimentales (Presidente de la Cámara, Secretario, etc.)
2. Normalizar apellido: minúsculas, sin acentos, sin ligaduras OCR (`ﬁ→fi`), sin contenido entre paréntesis
3. Cruzar por `apellido_norm` + `fecha` dentro del rango `bloque_inicio`–`bloque_fin`
4. **Match único** → asignar bloque. **Ambiguo o sin match** → descartar.

In [37]:
def normalizar_apellido(s):
    if not isinstance(s, str):
        return ""
    # Corregir ligaduras OCR
    s = s.replace("ﬁ", "fi").replace("ﬂ", "fl")
    # Quitar contenido entre paréntesis: "Martínez (Oscar Ariel)" → "Martínez"
    s = re.sub(r"\(.*?\)", "", s).strip()
    # Minúsculas y sin acentos
    s = s.lower().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.strip()

# Filtrar procedimentales
PROCEDIMENTALES = r"Presidente|Presidenta|Secretario|Secretaria|Jefe de Gabinete|Vicepresidente|Instructor|Presentador"
mask_proc = df["orador"].str.contains(PROCEDIMENTALES, na=False)
df_real = df[~mask_proc].copy()

df_real["apellido_norm"] = df_real["orador"].apply(normalizar_apellido)
dh["apellido_norm"]      = dh["apellido"].apply(normalizar_apellido)

print(f"Intervenciones después de filtrar procedimentales: {len(df_real):,}")

# JOIN: apellido + fecha dentro del rango bloque_inicio–bloque_fin
match_unico    = []
sin_match      = []
ambiguos       = []

for idx, row in df_real.iterrows():
    candidatos = dh[
        (dh["apellido_norm"]  == row["apellido_norm"]) &
        (dh["bloque_inicio"] <= row["fecha"]) &
        (dh["bloque_fin"]    >= row["fecha"])
    ]
    if len(candidatos) == 1:
        r = candidatos.iloc[0]
        match_unico.append({
            "idx": idx,
            "diputado_id":     r["id"],
            "nombre_completo": f"{r['nombre']} {r['apellido']}",
            "bloque":          r["bloque"],
            "familia_politica":r["familia_politica"],
            "distrito":        r["distrito"],
        })
    elif len(candidatos) == 0:
        sin_match.append(idx)
    else:
        ambiguos.append(idx)

total = len(df_real)
print(f"\nResultados del JOIN:")
print(f"  Match único:  {len(match_unico):,} ({len(match_unico)/total:.1%})")
print(f"  Ambiguo:      {len(ambiguos):,} ({len(ambiguos)/total:.1%})  → descartados")
print(f"  Sin match:    {len(sin_match):,} ({len(sin_match)/total:.1%})  → descartados")

Intervenciones después de filtrar procedimentales: 44,817

Resultados del JOIN:
  Match único:  37,987 (84.8%)
  Ambiguo:      1,559 (3.5%)  → descartados
  Sin match:    5,271 (11.8%)  → descartados


In [38]:
# Construir dataframe final con solo los matches únicos
df_matches = pd.DataFrame(match_unico).set_index("idx")
df_joined  = df_real.loc[df_matches.index].copy()
df_joined  = df_joined.join(df_matches[["diputado_id","nombre_completo","bloque","familia_politica","distrito"]])

print(f"Intervenciones con bloque asignado: {len(df_joined):,}")
print()
print("Distribución por familia política:")
print(df_joined["familia_politica"].value_counts().to_string())

Intervenciones con bloque asignado: 37,987

Distribución por familia política:
familia_politica
peronismo_kirchnerismo        12171
centro_derecha                 7271
radicalismo                    6001
peronismo_federal              4276
centro_izquierda               2502
provincial_federal             2362
izquierda                      2044
derecha_liberal_libertaria     1304
otros                            41
sin_bloque                       15


## 4. Selección de tópicos (mismos 10 que Experimentos 2 y 3)

In [39]:
TOPICOS_SELECCIONADOS = {
    1:   "Presupuesto / Finanzas públicas",
    6:   "Derecho penal",
    8:   "Impuestos / Fiscal",
    11:  "Trabajo / Laboral",
    12:  "Salud / Discapacidad",
    18:  "Jubilaciones / Previsional",
    5:   "Energía / Gas / Combustibles",
    129: "Derechos humanos / Terrorismo",
    30:  "Agropecuario / Ganadería",
    22:  "Defensa / Fuerzas militares",
}

df_validos = df_joined[df_joined["topic"].isin(TOPICOS_SELECCIONADOS.keys())].copy()
df_validos["topic_label"] = df_validos["topic"].map(TOPICOS_SELECCIONADOS)

# Familias con suficiente masa para analizar
FAMILIAS_MIN = 100
familias_ok = df_validos["familia_politica"].value_counts()
familias_ok = familias_ok[familias_ok >= FAMILIAS_MIN].index.tolist()
df_validos  = df_validos[df_validos["familia_politica"].isin(familias_ok)]

print(f"Intervenciones con tópico y familia política: {len(df_validos):,}")
print()
print("Intervenciones por familia política:")
print(df_validos["familia_politica"].value_counts().to_string())

Intervenciones con tópico y familia política: 5,431

Intervenciones por familia política:
familia_politica
peronismo_kirchnerismo        1665
centro_derecha                 924
radicalismo                    889
peronismo_federal              634
izquierda                      419
centro_izquierda               390
provincial_federal             323
derecha_liberal_libertaria     187


## 5. Heatmap: prevalencia de tópicos por familia política

In [ ]:
pivot = (
    df_validos
    .groupby(["familia_politica", "topic_label"])
    .size()
    .reset_index(name="count")
)
pivot["prop"] = pivot.groupby("familia_politica")["count"].transform(lambda x: x / x.sum())

heatmap_df = pivot.pivot(index="topic_label", columns="familia_politica", values="prop").fillna(0)

fig_heat = px.imshow(
    heatmap_df,
    labels=dict(x="Familia política", y="Tópico", color="Proporción"),
    title="Prevalencia de tópicos por familia política (2007–2026)",
    color_continuous_scale="Blues",
    aspect="auto",
    height=600,
)
fig_heat.update_xaxes(tickangle=90)
fig_heat.show()
fig_heat.write_html("figuras/exp4/heatmap_topicos_por_familia_politica.html")

## 6. Bar chart comparativo por familia política

In [ ]:
fig_bar = px.bar(
    pivot,
    x="topic_label",
    y="prop",
    color="familia_politica",
    barmode="group",
    title="Proporción de tópicos por familia política",
    labels={
        "topic_label":      "Tópico",
        "prop":             "Proporción de intervenciones",
        "familia_politica": "Familia política",
    },
    height=550,
    width=1300,
)
fig_bar.update_xaxes(tickangle=30)
fig_bar.update_layout(
    plot_bgcolor="white",
    legend_title_text="Familia política",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
)
fig_bar.show()
fig_bar.write_html("figuras/exp4/barras_topicos_por_familia_politica.html")

## 7. Test chi-cuadrado: ¿la distribución de tópicos depende de la familia política?

Tabla de contingencia: filas = familia política, columnas = tópico. El test evalúa si la asociación es estadísticamente significativa. El V de Cramér mide el tamaño del efecto (0 = sin asociación, 1 = asociación perfecta).

In [42]:
contingencia = pd.crosstab(df_validos["familia_politica"], df_validos["topic_label"])

chi2, p, dof, expected = chi2_contingency(contingencia)
n = contingencia.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingencia.shape) - 1)))

print(f"Chi-cuadrado: {chi2:.2f}")
print(f"p-valor:      {p:.2e}")
print(f"Grados de libertad: {dof}")
print(f"V de Cramér:  {cramers_v:.4f}")
print()
if p < 0.05:
    print("✅ La distribución de tópicos difiere significativamente entre familias políticas (p < 0.05)")
    if cramers_v < 0.1:
        print(f"   Tamaño del efecto: PEQUEÑO (V = {cramers_v:.3f})")
    elif cramers_v < 0.3:
        print(f"   Tamaño del efecto: MODERADO (V = {cramers_v:.3f})")
    else:
        print(f"   Tamaño del efecto: GRANDE (V = {cramers_v:.3f})")
else:
    print("❌ No hay evidencia de diferencia significativa entre familias políticas")

Chi-cuadrado: 376.49
p-valor:      3.42e-46
Grados de libertad: 63
V de Cramér:  0.0995

✅ La distribución de tópicos difiere significativamente entre familias políticas (p < 0.05)
   Tamaño del efecto: PEQUEÑO (V = 0.100)


## 8. TF-IDF por familia política: vocabulario distintivo

Tratamos todo el texto de cada familia como un único documento y calculamos TF-IDF para encontrar las palabras más características de cada sector.

In [43]:
# Palabras que se colaron pero no aportan información temática
STOPWORDS_EXTRA = {
    # Números y cantidades
    "ciento", "millón", "millones", "mil", "peso", "pesos",
    # Tiempo
    "año", "años", "mes", "meses", "dia", "dias",
    # Verbos y partículas genéricas
    "venir", "hacer", "decir", "tener", "poder", "deber", "ir",
    "ser", "estar", "haber", "dar", "ver", "querer", "saber",
    # Procedimentales que sobrevivieron la lematización
    "tema", "caso", "vez", "parte", "lugar", "punto",
    # Artículos / pronombres que sobrevivieron
    "el", "la", "lo", "le", "uno", "una",
}

# Concatenar textos por familia
textos_familia = (
    df_validos
    .groupby("familia_politica")["texto_limpio"]
    .apply(lambda x: " ".join(x.dropna()))
)

familias = textos_familia.index.tolist()
corpus   = textos_familia.values.tolist()

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    stop_words=list(STOPWORDS_EXTRA),
    token_pattern=r"(?u)\b[a-záéíóúüñ]{3,}\b",  # solo palabras de ≥3 letras
)
tfidf_matrix  = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()

N_TOP = 15
print(f"Top {N_TOP} palabras distintivas por familia política:\n")
for i, familia in enumerate(familias):
    scores  = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:N_TOP]
    top_words = [(feature_names[j], round(scores[j], 4)) for j in top_idx]
    print(f"── {familia} ──")
    print("  " + ", ".join(w for w, _ in top_words))
    print()

Top 15 palabras distintivas por familia política:

── centro_derecha ──
  presupuesto, gobierno, impuesto, país, provincia, público, argentina, trabajador, política, trabajo, sistema, social, argentino, persona, jubilado

── centro_izquierda ──
  presupuesto, gobierno, país, público, trabajador, social, sistema, argentina, política, trabajo, provincia, sector, empresa, situación, deuda

── derecha_liberal_libertaria ──
  impuesto, presupuesto, gobierno, país, público, inflación, provincia, fiscal, argentina, gasto, trabajador, sistema, generar, trabajo, argentino

── izquierda ──
  trabajador, gobierno, país, jubilado, presupuesto, pagar, frente, fondo, impuesto, acá, trabajo, laboral, seguir, provincia, derecho

── peronismo_federal ──
  presupuesto, gobierno, provincia, público, país, impuesto, sistema, argentina, política, social, trabajador, fondo, situación, económico, derecho

── peronismo_kirchnerismo ──
  presupuesto, gobierno, provincia, trabajador, país, trabajo, política, pú

In [ ]:
filas = []
for i, familia in enumerate(familias):
    scores  = tfidf_matrix[i].toarray().flatten()
    top_idx = scores.argsort()[::-1][:10]
    for j in top_idx:
        filas.append({"familia": familia, "palabra": feature_names[j], "tfidf": scores[j]})

df_tfidf = pd.DataFrame(filas)

from plotly.subplots import make_subplots

n_familias  = len(familias)
n_cols      = 3
n_rows      = -(-n_familias // n_cols)
row_height  = 280

fig_tfidf = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=familias,
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

colors = px.colors.qualitative.Set2

for i, familia in enumerate(familias):
    row = i // n_cols + 1
    col = i %  n_cols + 1
    sub = df_tfidf[df_tfidf["familia"] == familia].sort_values("tfidf")
    fig_tfidf.add_trace(
        go.Bar(
            x=sub["tfidf"],
            y=sub["palabra"],
            orientation="h",
            marker_color=colors[i % len(colors)],
            showlegend=False,
        ),
        row=row, col=col,
    )

fig_tfidf.update_layout(
    title_text="Vocabulario distintivo por familia política (TF-IDF, top 10 palabras)",
    height=n_rows * row_height,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig_tfidf.update_xaxes(showgrid=False)
fig_tfidf.update_yaxes(tickfont_size=11)

fig_tfidf.show()
fig_tfidf.write_html("figuras/exp4/tfidf_vocabulario_por_familia_politica.html")

## 9. Guardar resultados

In [46]:
# Guardar intervenciones con familia política asignada
df_validos[[
    "id_periodo","id_reunion","fecha","anio","orador","n_intervencion",
    "topic","topic_label","bloque","familia_politica","distrito"
]].to_parquet("../data/intervenciones_con_familia_politica.parquet", index=False)

# Guardar resultado del chi-cuadrado
pd.DataFrame([{
    "chi2": chi2, "p_valor": p, "dof": dof, "cramers_v": cramers_v
}]).to_csv("../data/chi2_familia_politica.csv", index=False)

print("Guardado: data/intervenciones_con_familia_politica.parquet")
print("Guardado: data/chi2_familia_politica.csv")

Guardado: data/intervenciones_con_familia_politica.parquet
Guardado: data/chi2_familia_politica.csv
